In [1]:
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"]="0"

import numpy as np
import tensorflow as tf
from tabulate import tabulate
from tensorflow.python.client import device_lib
from tqdm import tqdm
import time

2025-10-20 03:20:48.470150: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760910648.482949  296319 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760910648.486815  296319 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760910648.496909  296319 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760910648.496938  296319 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760910648.496940  296319 computation_placer.cc:177] computation placer alr

In [2]:

def print_gpu_details():
    """Prints all available GPUs in a formatted table with key details"""
    # Get list of all devices
    devices = device_lib.list_local_devices()
    gpu_details = []
    
    for device in devices:
        if device.device_type == 'GPU':
            # Extract details from the device description string
            desc = device.physical_device_desc
            details = {
                'Device ID': device.name.split(':')[-1],
                'Name': desc.split('name: ')[1].split(',')[0] if 'name: ' in desc else 'Unknown',
                'Memory (GB)': f"{device.memory_limit / (1024**3):.2f}",
                'PCI Bus ID': desc.split('pci bus id: ')[1].split(',')[0] if 'pci bus id: ' in desc else 'Unknown',
                'GFX Version': os.environ.get('HSA_OVERRIDE_GFX_VERSION', 'Native')
            }
            gpu_details.append(details)
    
    # Print table if GPUs found
    if gpu_details:
        print("\n" + "="*85)
        print("ACTIVE GPU CONFIGURATION".center(85))
        print("="*85)
        print(tabulate(gpu_details, headers="keys", tablefmt="grid"))
        print("="*85+ "\n")
    else:
        print("No GPU devices found!")


def configure_gpu(vram_limit):
    """Configure GPU settings with optional GFX override"""
    
    # Verify GPU availability
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        try:
            # Limit VRAM on the specified GPU
            
            for gid in range(len(gpus)):
                
                tf.config.experimental.set_virtual_device_configuration(
                    gpus[gid],
                    [tf.config.experimental.VirtualDeviceConfiguration(
                        memory_limit=vram_limit[gid] * 1024)]  # Convert GB to MB
                )
                print(f"GPU {gid} VRAM limited to {vram_limit[gid]}GB")
                
        except RuntimeError as e:
            print(f"Error setting VRAM limit: {e}")
    
    if not gpus:
        raise RuntimeError(f"No GPU found")
    
    for gid in range(len(gpus)):
        print(f"Configured GPU {gid}: {tf.config.experimental.get_device_details(gpus[gid])}") 
        
        with tf.device('/GPU:'+str(gid)):  # Force GPU usage
            x = tf.ones((1, 1))    # Smallest possible tensor
            y = x + 1              # Simple operation
            y.numpy()              # Force execution

In [3]:
VRAM = [3.0]

configure_gpu(VRAM)

print_gpu_details()

GPU 0 VRAM limited to 3.0GB
Configured GPU 0: {'compute_capability': (8, 6), 'device_name': 'NVIDIA GeForce RTX 3050 Laptop GPU'}

                               ACTIVE GPU CONFIGURATION                              
+-------------+------------------------------------+---------------+--------------+---------------+
|   Device ID | Name                               |   Memory (GB) | PCI Bus ID   | GFX Version   |
+=============+====================================+===============+==============+===============+
|           0 | NVIDIA GeForce RTX 3050 Laptop GPU |             3 | 0000:01:00.0 | Native        |
+-------------+------------------------------------+---------------+--------------+---------------+



I0000 00:00:1760910652.142447  296319 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3072 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1760910652.193268  296319 gpu_device.cc:2019] Created device /device:GPU:0 with 3072 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [ ]:

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models # type: ignore
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import Sequence # type: ignore
from tensorflow.keras.callbacks import ModelCheckpoint # type: ignore

# -------------------------
# Constants
# -------------------------

target = 400000
patch_shape = (50, 50)

Dmax = 800     # max disparity (width of right strip = n + Dmax)
BATCH_SIZE = 128
EPOCHS = 100


In [5]:

org_path = "/mnt/Extra/Project_Storage/stereo_ML_dataset/"
dataset_org_path = "/mnt/Extra/Project_Storage/stereo_ML_dataset/train/"

# -------------------------
# Load memmap datasets
# -------------------------
left_patch_memmap = dataset_org_path + "left_patch.dat"
right_strip_memmap = dataset_org_path + "right_strip.dat"
patch_disparity_memmap = dataset_org_path + "patch_disp.dat"

model_location = org_path+"model/"

left_patches  = np.memmap(left_patch_memmap, dtype=np.uint8, mode='r', shape=(target, patch_shape[0], patch_shape[1]))
right_strips  = np.memmap(right_strip_memmap, dtype=np.uint8, mode='r', shape=(target, patch_shape[0], patch_shape[1] + Dmax))
median_disp   = np.memmap(patch_disparity_memmap, dtype=np.int16, mode='r', shape=(target,))





In [6]:
from tensorflow.keras.utils import Sequence
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
import os

# -------------------------
# Memmap data generator (uses patch_shape)
# -------------------------
class MemmapGenerator(Sequence):
    def __init__(self, left_mm, right_mm, disp_mm, indices,
                 batch_size=8, patch_shape=(0, 0), Dmax=800, **kwargs):
        super().__init__(**kwargs)  # This allows multiprocessing arguments
        self.left_mm = left_mm
        self.right_mm = right_mm
        self.disp_mm = disp_mm
        self.indices = indices
        self.batch_size = batch_size
        self.patch_shape = patch_shape
        self.Dmax = Dmax

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        left_batch = self.left_mm[batch_idx]
        right_batch = self.right_mm[batch_idx]
        disp_batch = self.disp_mm[batch_idx].astype('float32') / self.Dmax

        h, w = self.patch_shape
        Xl = np.expand_dims(left_batch[:, :h, :w], axis=-1) / 255.0
        Xr = np.expand_dims(right_batch[:, :h, :w + self.Dmax], axis=-1) / 255.0

        return (Xl.astype('float32'), Xr.astype('float32')), disp_batch
    
# -------------------------
# Create training and validation generators
# -------------------------
all_indices = np.arange(target)
idx_train, idx_val = train_test_split(all_indices, test_size=0.1, random_state=42)

train_gen = MemmapGenerator(
    left_patches, right_strips, median_disp,
    idx_train, batch_size=BATCH_SIZE,
    patch_shape=patch_shape, Dmax=Dmax
)

val_gen = MemmapGenerator(
    left_patches, right_strips, median_disp,
    idx_val, batch_size=BATCH_SIZE,
    patch_shape=patch_shape, Dmax=Dmax
)



In [7]:
# -------------------------
# Encoder block
# -------------------------
def encoder_block(x):
    x = layers.Conv2D(16, (3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    return x

# -------------------------
# Build stereo model (uses patch_shape)
# -------------------------
def build_stereo_model_prob(patch_shape, Dmax=800):
    
    h, w = patch_shape

    # Define Inputs
    left_input  = layers.Input(shape=(h, w, 1), name='left_input')
    right_input = layers.Input(shape=(h, w + Dmax, 1), name='right_input')

    # Encode both sides
    feat_left  = encoder_block(left_input)
    feat_right = encoder_block(right_input)

    # Merge feature encodings
    combined = layers.Concatenate()([feat_left, feat_right])
    x = layers.Dense(64, activation='relu')(combined)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu')(x)

    # Sigmoid output (for normalized disparity)
    output = layers.Dense(1, activation='sigmoid')(x)

    # Build model
    model = models.Model(inputs=[left_input, right_input], outputs=output)
    return model


# -------------------------
# Checkpoint callback 
# -------------------------
import tensorflow as tf
import os

class Checkpoint(tf.keras.callbacks.Callback):
    def __init__(self, model_location, save_every=4):
        super().__init__()
        self.model_location = model_location
        self.save_every = save_every
        os.makedirs(model_location, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.save_every == 0:
            filename = os.path.join(self.model_location, f"model_epoch_{epoch+1:02d}.keras")
            self.model.save(filename)
            print(f"\nSaved model checkpoint: {filename}\n")

# -------------------------
# GPU Memory
# -------------------------

import tensorflow as tf

class GPUMemoryLogger(tf.keras.callbacks.Callback):
    def __init__(self, device_index=0):
        super().__init__()
        self.device_index = device_index

    def on_train_batch_end(self, batch, logs=None):
        try:
            # Get memory info for the chosen GPU
            mem_info = tf.config.experimental.get_memory_info(f'GPU:{self.device_index}')
            used = mem_info['current'] / (1024 ** 2)  # Convert to MB
            peak = mem_info['peak'] / (1024 ** 2)     # Convert to MB
            print(f"Batch {batch}: GPU{self.device_index} memory used = {used:.1f} MB | peak = {peak:.1f} MB")
        except Exception as e:
            print(f"Could not fetch GPU info: {e}")


# -------------------------
# Compile and attach callback
# -------------------------
model = build_stereo_model_prob(patch_shape=patch_shape, Dmax=Dmax)
model.compile(optimizer='adam', loss='mae', metrics=['mse'])
model.summary()

checkpoint_cb = Checkpoint(model_location, save_every=5)
gpu_logger = GPUMemoryLogger()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ left_input          │ (None, 50, 50, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ right_input         │ (None, 50, 850,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 50, 50,    │        160 │ left_input[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 50, 850,   │        160 │ right_input[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 50, 50,    │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 850,   │         64 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 25, 25,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 25, 425,   │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 25, 25,    │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 25, 425,   │      4,640 │ max_pooling2d_2[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 25,    │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 25, 425,   │        128 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 12, 12,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 12, 212,   │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 12, 12,    │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 12, 212,   │     18,496 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 12, 12,    │        256 │ conv2d_2[0][0]  

 Total params: 57,857 (226.00 KB)

 Trainable params: 57,409 (224.25 KB)

 Non-trainable params: 448 (1.75 KB)

In [8]:
# -------------------------
# Train (multiprocessing enabled)
# -------------------------
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    verbose=1,
    callbacks=[checkpoint_cb]
)

Epoch 1/100


I0000 00:00:1760910654.822036  296413 service.cc:152] XLA service 0x7e73d801c040 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1760910654.822063  296413 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2025-10-20 03:20:54.891412: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1760910655.299778  296413 cuda_dnn.cc:529] Loaded cuDNN version 90701


   1/2813 ━━━━━━━━━━━━━━━━━━━━ 11:18:30 14s/step - loss: 0.2316 - mse: 0.0684

I0000 00:00:1760910667.226419  296413 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2813/2813 ━━━━━━━━━━━━━━━━━━━━ 336s 114ms/step - loss: 0.0536 - mse: 0.0087 - val_loss: 0.1243 - val_mse: 0.0407
Epoch 2/100
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 312s 111ms/step - loss: 0.0274 - mse: 0.0036 - val_loss: 0.0695 - val_mse: 0.0141
Epoch 3/100
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 310s 110ms/step - loss: 0.0227 - mse: 0.0030 - val_loss: 0.1150 - val_mse: 0.0259
Epoch 4/100
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 315s 112ms/step - loss: 0.0201 - mse: 0.0027 - val_loss: 0.1195 - val_mse: 0.0351
Epoch 5/100
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 0.0189 - mse: 0.0025
Saved model checkpoint: /mnt/Extra/Project_Storage/stereo_ML_dataset/model/model_epoch_05.keras

2813/2813 ━━━━━━━━━━━━━━━━━━━━ 314s 112ms/step - loss: 0.0184 - mse: 0.0024 - val_loss: 0.0322 - val_mse: 0.0061
Epoch 6/100
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 313s 111ms/step - loss: 0.0172 - mse: 0.0023 - val_loss: 0.0239 - val_mse: 0.0038
Epoch 7/100
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 315s 112ms/step - loss: 0.0164 - mse: 0.0022 - v

In [9]:
import numpy as np

# Save
np.save(model_location+"history.npy", history.history)
